# Project Sanjay MK2 -- Day 4: Thermal YOLO Training

**Goal:** Train a thermal-specific YOLO11s model on aerial IR imagery. This is the second half of the sensor-adaptive perception architecture (RGB model already trained as `police_full_v2.pt`).

**Prerequisite for:** SensorScheduler RL training (needs both RGB and thermal models).

| Item | Detail |
|------|--------|
| Dataset | HIT-UAV (~2,866 images) + M3OT IR (~10,790 images) = **~13,656 aerial thermal images, ~135K boxes** |
| Classes trained | person (0), vehicle (2) -- other classes unused but slot-compatible with RGB model |
| Init | yolo11s.pt (COCO-pretrained, NOT police_full_v2 -- RGB->thermal domain shift) |
| Training | 150 epochs, thermal-specific augmentations |
| Output | thermal_police.pt at Drive/SanjayMK2/runs/thermal_police/weights/ |
| Auto-resume | yes (same pattern as Day 3) |

**Datasets verified via Day 4a inspection:**
- HIT-UAV: YOLO format, 5 classes (0=Person, 1=Car, 2=Bicycle, 3=OtherVehicle, 4=DontCare) → remap {0:0, 1/2/3:2, 4:drop}
- M3OT IR: COCO JSON, single category 'vehicle' → all → our class 2

## CELL 1 -- Install + clone repo

In [ ]:
!pip install ultralytics kagglehub -q
!git clone https://github.com/aniket-27507/Sanjay_MK2 /content/Sanjay_MK2
%cd /content/Sanjay_MK2

# Write thermal dataset YAML (same 6-class scheme as RGB for downstream compatibility)
import os
os.makedirs('config/training', exist_ok=True)
with open('config/training/thermal_police.yaml', 'w') as f:
    f.write(
        "path: /content/Sanjay_MK2/data/thermal_training\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: person\n"
        "  1: weapon_person\n"
        "  2: vehicle\n"
        "  3: fire\n"
        "  4: explosive_device\n"
        "  5: crowd\n"
    )
print('Thermal YAML written')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## CELL 2 -- Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
drive_path = '/content/drive/MyDrive/SanjayMK2'
os.makedirs(f'{drive_path}/runs', exist_ok=True)
os.makedirs(f'{drive_path}/datasets', exist_ok=True)
print(f'Drive SanjayMK2 dir: {drive_path}')

## CELL 2b -- Kaggle credentials (required for Cell 3 HIT-UAV download)

HIT-UAV is hosted on Kaggle, so `kagglehub` in Cell 3 needs a Kaggle API token.

**First-time setup (do this once):**
1. Go to https://www.kaggle.com -> avatar (top-right) -> **Settings** -> **API** -> **Create New Token**.
2. Your browser downloads `kaggle.json` (contains `username` + `key`).
3. Run the cell below. When the upload dialog appears, pick that `kaggle.json`.

The cell caches the file to `Drive/SanjayMK2/.kaggle/kaggle.json` so future Colab sessions reuse it automatically -- you only upload once per token rotation.

**Note:** The key never appears in the notebook source; it's only read at runtime from the uploaded file. Do not paste the key as text into any cell.

In [ ]:
import json, shutil
from pathlib import Path

KAGGLE_DIR   = Path.home() / '.kaggle'
KAGGLE_JSON  = KAGGLE_DIR / 'kaggle.json'
DRIVE_KAGGLE = Path('/content/drive/MyDrive/SanjayMK2/.kaggle/kaggle.json')

KAGGLE_DIR.mkdir(parents=True, exist_ok=True)

if KAGGLE_JSON.exists():
    print(f'kaggle.json already at {KAGGLE_JSON} (runtime cache hit)')
elif DRIVE_KAGGLE.exists():
    shutil.copy(DRIVE_KAGGLE, KAGGLE_JSON)
    print(f'Loaded kaggle.json from Drive: {DRIVE_KAGGLE}')
else:
    print('No cached kaggle.json found.')
    print('Upload the kaggle.json you downloaded from kaggle.com -> Settings -> API -> Create New Token')
    from google.colab import files
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('Expected a file named exactly "kaggle.json"')
    shutil.move('kaggle.json', KAGGLE_JSON)
    DRIVE_KAGGLE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(KAGGLE_JSON, DRIVE_KAGGLE)
    print(f'Cached on Drive at {DRIVE_KAGGLE} (future sessions will reuse)')

KAGGLE_JSON.chmod(0o600)

with open(KAGGLE_JSON) as f:
    creds = json.load(f)
assert 'username' in creds and 'key' in creds, 'kaggle.json is missing username or key'
print(f'Kaggle credentials ready (user: {creds["username"]})')

## CELL 3 -- Download + process HIT-UAV (~2,866 images)

HIT-UAV classes: 0=Person, 1=Car, 2=Bicycle, 3=OtherVehicle, 4=DontCare

Remap:
- 0 (Person) → 0 (person)
- 1/2/3 (Car/Bicycle/OtherVehicle) → 2 (vehicle)
- 4 (DontCare) → dropped

Tested locally: 2008 train + 858 val images, 12312 person + 12439 vehicle boxes.

In [ ]:
import kagglehub, shutil
from pathlib import Path

DST = Path('data/thermal_training')

# Check if HIT-UAV already processed
existing_hituav = 0
if (DST/'images'/'train').exists():
    existing_hituav = len([f for f in (DST/'images'/'train').glob('hituav_*')])

if existing_hituav > 1500:
    print(f'HIT-UAV already processed ({existing_hituav} files). Skipping.')
else:
    print('Downloading HIT-UAV from Kaggle...')
    src_root = Path(kagglehub.dataset_download('pandrii000/hituav-a-highaltitude-infrared-thermal-dataset'))
    print(f'  Downloaded to: {src_root}')

    src = src_root / 'hit-uav'
    if not src.exists():
        src = src_root

    REMAP = {0: 0, 1: 2, 2: 2, 3: 2}  # 4 (DontCare) dropped
    counts = {'train': 0, 'val': 0}
    box_counts = {}

    for src_split in ['train', 'val', 'test']:
        dst_split = 'val' if src_split in ('val', 'test') else 'train'
        src_img = src / 'images' / src_split
        src_lbl = src / 'labels' / src_split
        dst_img = DST / 'images' / dst_split
        dst_lbl = DST / 'labels' / dst_split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)

        if not src_lbl.exists():
            continue

        for lbl_path in src_lbl.glob('*.txt'):
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                candidate = src_img / (lbl_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if not img_path:
                continue

            lines = []
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    old = int(parts[0])
                    new = REMAP.get(old)
                    if new is not None:
                        parts[0] = str(new)
                        lines.append(' '.join(parts))
                        box_counts[new] = box_counts.get(new, 0) + 1

            if not lines:
                continue

            dst_i = dst_img / f'hituav_{img_path.name}'
            dst_l = dst_lbl / f'hituav_{lbl_path.stem}.txt'
            if not dst_i.exists():
                shutil.copy2(img_path, dst_i)
            if not dst_l.exists():
                dst_l.write_text('\n'.join(lines) + '\n')
            counts[dst_split] += 1

    print(f'\nHIT-UAV processed:')
    for s, n in counts.items():
        print(f'  {s}: {n} images')
    print(f'  Box counts: {box_counts}')

## CELL 4 -- Process M3OT IR (stream from cached ZIP on Drive)

M3OT is the 10.7 GB ZIP cached to Drive in Day 4a. Stream IR images directly from ZIP (no full extraction).

**Structure verified:**
- IR images at: `M3OT/{drone}/ir/{split}/{sequence}/img1/{file}.PNG`
- COCO annotations at: `M3OT/Annotations/{drone}/ir/{split}_cocoformat.json`
- Single category: `{id: 1, name: 'vehicle'}` → remap to our class 2
- Image size: 640×512 RGB mode (IR stored as RGB channels)

**Expected output:** ~10,790 images, ~110,000 vehicle boxes. Runtime: ~10-15 min.

In [ ]:
import zipfile, json, shutil, os
from pathlib import Path
from collections import defaultdict

DST = Path('data/thermal_training')
M3OT_ZIP = '/content/drive/MyDrive/SanjayMK2/datasets/M3OT.zip'

# If not cached on Drive, download from Figshare
if not os.path.exists(M3OT_ZIP):
    print('M3OT not on Drive. Downloading from Figshare (10.7 GB, ~10-15 min)...')
    os.makedirs(os.path.dirname(M3OT_ZIP), exist_ok=True)
    !wget -O {M3OT_ZIP} --show-progress 'https://ndownloader.figshare.com/files/52023875'
else:
    print(f'Using cached M3OT ZIP: {os.path.getsize(M3OT_ZIP) / 1e9:.2f} GB')

# Skip if already processed
existing = 0
if (DST/'images'/'train').exists():
    existing = len([f for f in (DST/'images'/'train').glob('m3ot_*')])

if existing > 5000:
    print(f'M3OT already processed ({existing} files). Skipping.')
else:
    counts = {'train': 0, 'val': 0}
    boxes_added = 0
    skipped_no_img = 0

    with zipfile.ZipFile(M3OT_ZIP, 'r') as z:
        zip_paths = set(z.namelist())
        print(f'ZIP contains {len(zip_paths)} entries')

        for drone in ['1', '2']:
            for src_split in ['train', 'val', 'test']:
                dst_split = 'val' if src_split in ('val', 'test') else 'train'

                coco_path = f'M3OT/Annotations/{drone}/ir/{src_split}_cocoformat.json'
                if coco_path not in zip_paths:
                    continue

                with z.open(coco_path) as f:
                    coco = json.loads(f.read())

                img_info = {im['id']: im for im in coco['images']}
                ann_by_img = defaultdict(list)
                for ann in coco['annotations']:
                    ann_by_img[ann['image_id']].append(ann)

                print(f'\n  Drone {drone} / {src_split}: {len(img_info)} images, {len(coco["annotations"])} annotations')

                for img_id, anns in ann_by_img.items():
                    info = img_info.get(img_id)
                    if not info:
                        continue
                    W, H = info['width'], info['height']
                    if W <= 0 or H <= 0:
                        continue

                    # Parse file_name: '/home/dell/nzh/dataset/m3ot-t/train/1-04T/img1/000001.PNG'
                    fname = info['file_name']
                    parts = fname.split('/')
                    try:
                        img1_idx = parts.index('img1')
                        seq = parts[img1_idx - 1]
                        filename = parts[-1]
                    except (ValueError, IndexError):
                        continue

                    zip_img_path = f'M3OT/{drone}/ir/{src_split}/{seq}/img1/{filename}'
                    if zip_img_path not in zip_paths:
                        skipped_no_img += 1
                        continue

                    # Convert COCO bboxes to YOLO (all class 2 = vehicle)
                    lines = []
                    for ann in anns:
                        x, y, bw, bh = ann['bbox']
                        cx = (x + bw/2) / W
                        cy = (y + bh/2) / H
                        nw = bw / W
                        nh = bh / H
                        cx, cy = max(0, min(1, cx)), max(0, min(1, cy))
                        nw, nh = max(0, min(1, nw)), max(0, min(1, nh))
                        if nw > 0.001 and nh > 0.001:
                            lines.append(f'2 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
                            boxes_added += 1

                    if not lines:
                        continue

                    stem = Path(filename).stem
                    out_name = f'm3ot_d{drone}_{seq}_{stem}'
                    dst_img = DST / 'images' / dst_split / f'{out_name}.png'
                    dst_lbl = DST / 'labels' / dst_split / f'{out_name}.txt'
                    dst_img.parent.mkdir(parents=True, exist_ok=True)
                    dst_lbl.parent.mkdir(parents=True, exist_ok=True)

                    if not dst_img.exists():
                        with z.open(zip_img_path) as src_f, open(dst_img, 'wb') as dst_f:
                            shutil.copyfileobj(src_f, dst_f)
                    if not dst_lbl.exists():
                        dst_lbl.write_text('\n'.join(lines) + '\n')

                    counts[dst_split] += 1
                    if counts[dst_split] % 1000 == 0:
                        print(f'    ... {counts[dst_split]} images ({dst_split})')

    print(f'\n{"=" * 60}')
    print(f'M3OT IR processing complete:')
    print(f'  Train: {counts["train"]} images')
    print(f'  Val:   {counts["val"]} images')
    print(f'  Vehicle boxes: {boxes_added}')
    print(f'  Skipped (no matching image): {skipped_no_img}')
    print(f'{"=" * 60}')

## CELL 5 -- Final merged dataset sanity check

Expected after HIT-UAV + M3OT:
- Train: ~10,000 images
- Val: ~3,500 images  
- Box counts: {0 (person): ~12K, 2 (vehicle): ~125K}

In [ ]:
from pathlib import Path
from collections import Counter

DST = Path('data/thermal_training')
print('=' * 60)
print('FINAL MERGED THERMAL DATASET')
print('=' * 60)

for split in ['train', 'val']:
    img_dir = DST / 'images' / split
    lbl_dir = DST / 'labels' / split
    all_imgs = list(img_dir.glob('*')) if img_dir.exists() else []
    hituav = [f for f in all_imgs if f.name.startswith('hituav_')]
    m3ot = [f for f in all_imgs if f.name.startswith('m3ot_')]

    c = Counter()
    if lbl_dir.exists():
        for lp in lbl_dir.glob('*.txt'):
            for line in lp.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    c[int(parts[0])] += 1

    print(f'\n{split}:')
    print(f'  Total images: {len(all_imgs)}')
    print(f'    HIT-UAV: {len(hituav)}')
    print(f'    M3OT:    {len(m3ot)}')
    print(f'  Box counts: {dict(sorted(c.items()))}')
    print(f'    (class 0 = person, class 2 = vehicle)')

## CELL 6 -- Train thermal YOLO11s (150 epochs, auto-resumable)

Key differences from RGB training:
- **Init:** `yolo11s.pt` (COCO-pretrained), NOT police_full_v2 (RGB→thermal domain shift)
- **No HSV augmentation** (`hsv_h=0, hsv_s=0`) -- grayscale IR has no color
- **No vertical flip** (`flipud=0`) -- aerial has gravity
- **150 epochs** -- more than RGB (thermal has lower feature diversity)
- **Auto-resume** from `last.pt` on Drive

**Training time:** ~12-18 hours on T4 (~13.6K images × 150 epochs, ~5-6 min/epoch)

In [ ]:
import os
from ultralytics import YOLO

DRIVE_PROJECT = '/content/drive/MyDrive/SanjayMK2/runs'
RUN_NAME = 'thermal_police'
LAST_PT = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

if os.path.exists(LAST_PT):
    print(f'Resuming from: {LAST_PT}')
    print(f'  Checkpoint size: {os.path.getsize(LAST_PT) / 1e6:.1f} MB')
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
else:
    print('Starting fresh thermal training from yolo11s.pt')
    init_weights = 'yolo11s.pt' if os.path.exists('yolo11s.pt') else 'yolo11s.pt'
    model = YOLO(init_weights)
    results = model.train(
        data='config/training/thermal_police.yaml',
        epochs=150,
        imgsz=640,
        device=0,
        batch=-1,
        patience=30,
        name=RUN_NAME,
        project=DRIVE_PROJECT,
        exist_ok=True,
        # Thermal-specific augmentations
        hsv_h=0.0,      # no hue (grayscale)
        hsv_s=0.0,      # no saturation
        hsv_v=0.3,      # gain variation OK
        mosaic=1.0,
        mixup=0.1,
        fliplr=0.5,
        flipud=0.0,     # no vertical flip (aerial has gravity)
        degrees=10.0,
        scale=0.5,
        translate=0.1,
    )

print('\n=== Training complete ===')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)'))
print(f'Weights on Drive: {DRIVE_PROJECT}/{RUN_NAME}/weights/')

## CELL 7 -- Download best.pt

After training completes, download `best.pt` and copy locally to:
```
runs/detect/runs/detect/thermal_police/weights/best.pt
```

This becomes the thermal detection backend for the SensorScheduler (Day 5).

In [ ]:
from google.colab import files
import os

best_path = '/content/drive/MyDrive/SanjayMK2/runs/thermal_police/weights/best.pt'
if os.path.exists(best_path):
    print(f'Best weights: {os.path.getsize(best_path) / 1e6:.1f} MB')
    files.download(best_path)
else:
    print(f'WARNING: {best_path} not found')

print()
print('Next steps:')
print('  1. Copy best.pt to: runs/detect/runs/detect/thermal_police/weights/best.pt')
print('  2. Day 5: implement src/single_drone/sensor_scheduler.py')

## Day 4 Success Criteria

| Class | Target mAP50 | Notes |
|-------|--------------|-------|
| person | > 0.60 | HIT-UAV has 12K+ person boxes |
| vehicle | > 0.60 | HIT-UAV (12K) + M3OT (110K) = ~125K boxes |
| overall | > 0.60 | 2-class model with ~135K total boxes |

## Data sources summary

| Source | Images | Boxes | Classes | License |
|--------|--------|-------|---------|---------|
| HIT-UAV | ~2,866 | ~24,751 | Person (12K) + Vehicle (12K) | CC BY-NC-SA 4.0 |
| M3OT (IR only) | ~10,790 | ~110,000 | Vehicle only | CC BY 4.0 |
| **Total** | **~13,656** | **~135,000** | person + vehicle | |

## Notes on thermal fire

FLAME 2 thermal fire dataset requires Kaggle consent form. Skipped for v1.
Fire detection stays primarily on RGB (police_full_v2 scored 0.762 mAP50 on fire).

## Architecture role

This thermal model is the second sensor backend for the SensorScheduler (Day 5):
- **Primary at night** (when RGB is dimmed to 5 FPS)
- **Triggered during day** for occlusion scenarios (foliage, tinted vehicles)
- **Cross-modal verification** when RGB confidence drops
- NOT a replacement for police_full_v2 (RGB handles all 6 classes)